# MLOPS Assignment 1

Student: 2025AE05188

## 02. Feature Engineering & Model Development

In this section, the UCI Heart Disease dataset is prepared for machine learning.

The target variable is converted into a binary classification problem, where `0` represents absence of heart disease and `1` represents presence of heart disease.

The input features are divided into numerical and categorical features. Numerical features are scaled for Logistic Regression, while categorical features are one-hot encoded.

Two baseline classification models are considered:

- Logistic Regression as a linear baseline.
- Random Forest as a nonlinear tree-based model.

The models are evaluated using five-fold stratified cross-validation on the training data and then evaluated on an unseen test set using accuracy, precision, recall, and ROC-AUC.

The baseline models use default scikit-learn hyperparameters. Hyperparameter tuning and further experimentation are planned for Section 3 using MLflow.

#### Imports

In [1]:
!pip3 install -U ucimlrepo

In [42]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate

### Heart Disease UCI Dataset

In [3]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features.copy()
y = heart_disease.data.targets.copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (303, 13)
Target shape: (303, 1)


In [4]:
display(X.head())

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0


In [5]:
display(y.head())

,num
0,0
1,2
2,1
3,0
4,0


In [6]:
print("Feature columns:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.columns.tolist())

print("\nTarget values:")
print(y.iloc[:, 0].value_counts().sort_index())

Feature columns:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

Target column:
['num']

Target values:
num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64


Make the target binary for the classification.

In [7]:
target_column = y.columns[0]

y = (y[target_column] > 0).astype(int)

print(y.value_counts().sort_index())

num
0    164
1    139
Name: count, dtype: int64


### Target Transformation

The original UCI target contains values from `0` to `4`, where `0` indicates absence of heart disease and values `1–4` indicate different levels of disease presence.

For this assignment, the problem is converted into binary classification:

- `0` → No heart disease
- `1–4` → Heart disease present

Therefore, the target is transformed using:

`target = 1` when the original value is greater than 0, otherwise `target = 0`.

### Train-Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Training samples: 242
Test samples: 61

Training target distribution:
num
0    0.541322
1    0.458678
Name: proportion, dtype: float64

Test target distribution:
num
0    0.540984
1    0.459016
Name: proportion, dtype: float64


The dataset is divided into 80% training data and 20% test data.

Stratified splitting is used to preserve approximately the same proportion of positive and negative cases in both sets.

The test set will be kept untouched during preprocessing, model training, and cross-validation. It is used only for the final evaluation of the baseline models.

#### Input features (Categorical and Numeric)

In [11]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak",
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal",
]

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categorical features: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']


In [12]:
assert set(numerical_features + categorical_features) == set(X.columns)

Why ca is categorical here??

Although UCI describes it as an integer variable, ca takes a small number of discrete values representing the number of major vessels.

We don't want the model to assume that all integer-coded categorical variables represent continuous quantities.

### Model Building and pipeline set up

Model-Specific Preprocessing:

The same categorical encoding strategy is used for both models, but numerical scaling is handled differently.

Logistic Regression benefits from standardized numerical features because it is a distance/scale-sensitive linear optimization model.

Random Forest is based on decision trees and does not require numerical feature scaling. Therefore, numerical features are imputed but kept on their original scale.

Both preprocessing steps are implemented inside scikit-learn pipelines so that preprocessing is fitted only on the training data and is applied consistently during cross-validation and testing.

For Logistic Regression:

In [24]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [25]:
logistic_regression = Pipeline(
    steps=[
        ("preprocessor", preprocessor_lr),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

For Random Forest:

In [26]:
numeric_transformer_rf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer_rf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_rf, numerical_features),
        ("cat", categorical_transformer_rf, categorical_features),
    ]
)

In [27]:
random_forest = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),
        (
            "model",
            RandomForestClassifier(
                random_state=42,
            ),
        ),
    ]
)

### Train and Evaluate Logistic Regression and Random Forest

Train the models with pipeline

In [28]:
logistic_regression.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'trestbps', 'chol',
                                                   'thalach', 'oldpeak']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['sex', 'cp', 'fbs',
                                                   'restecg', 'exang', 'slope',
                                                   'ca', 'thal'])])),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [29]:
random_forest.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'trestbps', 'chol',
                                                   'thalach', 'oldpeak']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['sex', 'cp', 'fbs',
                                                   'restecg', 'exang', 'slope',
                                                   'ca', 'thal'])])),
                ('model', RandomForestClassifier(random_state=42))])

Evaluiate the models

In [39]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }

In [40]:
lr_test_metrics = evaluate_model(
    logistic_regression,
    X_test,
    y_test,
)

rf_test_metrics = evaluate_model(
    random_forest,
    X_test,
    y_test,
)

In [41]:
test_results = pd.DataFrame(
    {
        "Logistic Regression": lr_test_metrics,
        "Random Forest": rf_test_metrics,
    }
).T

display(test_results.round(4))

,accuracy,precision,F1-score,recall,roc_auc
Logistic Regression,0.8852,0.8387,0.8814,0.9286,0.9665
Random Forest,0.8689,0.8125,0.8667,0.9286,0.9421


We can see that logistic regression performs better than the random forest in terms of accuracy, precision, f1-score and roc_auc score. Recall is almost same.

On the held-out test set, Logistic Regression currently shows higher performance than Random Forest across the observed evaluation metrics.

However, this comparison is based on a single test split. Therefore, cross-validation is used next to examine whether the observed performance is consistent across different subsets of the training data.

The difference in performance will be interpreted together with the cross-validation results rather than relying only on the single test-set result.

### Train and Evaluate Logistic Regression and Random Forest with CV

A single train-test split provides an estimate of model performance on one particular held-out subset. To obtain a more reliable estimate of baseline model performance and assess its consistency across different subsets of the training data, five-fold stratified cross-validation is performed.

Stratification preserves the class distribution across the folds.

The test set remains untouched during cross-validation and is used only for the final evaluation.

In [43]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [45]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

In [46]:
lr_cv = cross_validate(
    logistic_regression,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
)

rf_cv = cross_validate(
    random_forest,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
)

#### Summarize CV results

In [47]:
def summarize_cv_results(cv_results):
    metrics = {}

    for metric in scoring:
        values = cv_results[f"test_{metric}"]
        metrics[metric] = {
            "mean": values.mean(),
            "std": values.std(),
        }

    return metrics

In [49]:
lr_cv_summary = summarize_cv_results(lr_cv)
rf_cv_summary = summarize_cv_results(rf_cv)

In [50]:
print("Logistic Regression CV:")
display(pd.DataFrame(lr_cv_summary).T)

Logistic Regression CV:


,mean,std
accuracy,0.847109,0.009977
precision,0.876753,0.054190
recall,0.783399,0.045918
f1,0.824501,0.006867
roc_auc,0.902493,0.014440


In [51]:
print("Random Forest CV:")
display(pd.DataFrame(rf_cv_summary).T)

Random Forest CV:


,mean,std
accuracy,0.818112,0.030707
precision,0.817253,0.040538
recall,0.782609,0.098108
f1,0.794756,0.046911
roc_auc,0.896697,0.028026


Five-fold stratified cross-validation is performed on the training data to estimate the stability of each baseline model across different subsets of the training data.

After cross-validation, each model is fitted on the complete training set and evaluated once on the untouched test set.

Cross-validation is used to understand model consistency, while the test-set results provide the final unbiased evaluation on unseen data.

In [56]:
comparison = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Random Forest",
        ],
        "CV Accuracy": [
            lr_cv["test_accuracy"].mean(),
            rf_cv["test_accuracy"].mean(),
        ],
        "CV Precision": [
            lr_cv["test_precision"].mean(),
            rf_cv["test_precision"].mean(),
        ],
        "CV Recall": [
            lr_cv["test_recall"].mean(),
            rf_cv["test_recall"].mean(),
        ],
        "CV F1": [
            lr_cv["test_f1"].mean(),
            rf_cv["test_f1"].mean(),
        ],
        "CV ROC-AUC": [
            lr_cv["test_roc_auc"].mean(),
            rf_cv["test_roc_auc"].mean(),
        ],
        "Test Accuracy": [
            lr_test_metrics["accuracy"],
            rf_test_metrics["accuracy"],
        ],
        "Test Precision": [
            lr_test_metrics["precision"],
            rf_test_metrics["precision"],
        ],
        "Test Recall": [
            lr_test_metrics["recall"],
            rf_test_metrics["recall"],
        ],
        "Test F1": [
            lr_test_metrics["F1-score"],
            rf_test_metrics["F1-score"],
        ],
        "Test ROC-AUC": [
            lr_test_metrics["roc_auc"],
            rf_test_metrics["roc_auc"],
        ],
    }
)

display(comparison.round(4))

,Model,CV Accuracy,CV Precision,CV Recall,CV F1,CV ROC-AUC,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC
0,Logistic Regression,0.8471,0.8768,0.7834,0.8245,0.9025,0.8852,0.8387,0.9286,0.8814,0.9665
1,Random Forest,0.8181,0.8173,0.7826,0.7948,0.8967,0.8689,0.8125,0.9286,0.8667,0.9421


#### Summary

Two baseline classification models were developed: Logistic Regression and Random Forest.

Both models use preprocessing pipelines to handle missing values and categorical features. Numerical features are standardized for Logistic Regression, while Random Forest uses the numerical features without scaling because tree-based models do not require feature scaling.

Five-fold stratified cross-validation was used on the training data, followed by evaluation on an unseen test set using accuracy, precision, recall, and ROC-AUC.

From the obtained cross-validation and test results, Logistic Regression shows higher baseline performance across the evaluated metrics compared with Random Forest.

These results represent the performance of the initial default-parameter configurations and should therefore be treated as baseline results rather than final model performance.

No hyperparameter tuning or feature-selection experiments were performed at this stage. The models and preprocessing pipelines established here will serve as baselines for the subsequent MLflow experimentation stage.

In Section 3, the selected models will be systematically experimented with through hyperparameter tuning and alternative feature/preprocessing configurations where appropriate.

On the selected models, experiments can be performed for hyper-parameter tuning and feature engineering can be also done to improve the prediction capability of the models.

### Model Selection and Tuning Plan

Based on the EDA findings and the nature of the dataset, two complementary baseline models are selected:

- **Logistic Regression** — selected as a simple, interpretable linear baseline.
- **Random Forest** — selected as a nonlinear tree-based model capable of capturing feature interactions.

For this section, both models are trained using the **default scikit-learn model parameters**, with only `random_state` set for reproducibility and `max_iter=1000` for Logistic Regression to ensure convergence.

The baseline models will establish the initial performance using accuracy, precision, recall, and ROC-AUC.

In **Section 3 (Experiment Tracking)**, these baseline models will be systematically experimented with using MLflow. The planned experiments will include:

- Hyperparameter tuning for Logistic Regression, primarily exploring `C` and potentially regularization-related settings.
- Hyperparameter tuning for Random Forest (e.g., `n_estimators`, `max_depth`, `min_samples_split`, `max_features`, and `min_samples_leaf`).
- Comparison of different model configurations using cross-validation.
- Tracking parameters, metrics, and relevant artifacts for each experiment in MLflow.
- Comparing the experiments against the baseline results to identify a suitable final model configuration.

Thus, the models and default configurations established here serve as the **baseline for the systematic experimentation and tuning performed in Section 3**.